In [13]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import nltk

In [11]:
# Download NLTK punkt tokenizer
nltk.download('punkt')

# Load LLaMA tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.61s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [ ]:
def generate_and_save_rdf(prompt, top_k=3, n_steps=5, output_file="branching_structure.ttl"):
    # Initialize an RDF graph and namespace
    g = Graph()
    EX = Namespace("http://example.org/")
    g.bind("ex", EX)

    # Tokenize the initial prompt
    tokens = tokenizer(prompt, return_tensors='pt')
    input_ids = tokens.input_ids

    # Track each token generated in each step
    current_nodes = [input_ids]  # Start with the initial prompt as the root node

    for step in range(n_steps):
        next_nodes = []  # Reset next_nodes for each step to only hold new branches

        for branch_id, branch_input_ids in enumerate(current_nodes):
            # Generate top-k next tokens for each current token/branch
            with torch.no_grad():
                outputs = model(branch_input_ids)
                logits = outputs.logits[:, -1, :]

                # Ensure exactly top_k next tokens are selected
                top_k_probs, top_k_indices = torch.topk(logits, top_k, dim=-1)

                # Iterate over each of the top-k options and create branches
                for k in range(top_k):
                    next_token = top_k_indices[0, k].item()
                    next_token_text = tokenizer.decode([next_token], skip_special_tokens=True)
                    new_input_ids = torch.cat([branch_input_ids, torch.tensor([[next_token]])], dim=-1)

                    # Create URIs for the current and next nodes
                    current_uri = EX[f"Branch_{step}_{branch_id}"]
                    next_uri = EX[f"Branch_{step+1}_{branch_id}_{k}"]

                    # Add current token's branch node if not already present
                    if (current_uri, RDF.type, EX.Token) not in g:
                        g.add((current_uri, RDF.type, EX.Token))
                        g.add((current_uri, EX.text, Literal(tokenizer.decode(branch_input_ids[0], skip_special_tokens=True))))
                        g.add((current_uri, EX.iteration, Literal(step)))

                    # Add next token node
                    g.add((next_uri, RDF.type, EX.Token))
                    g.add((next_uri, EX.text, Literal(next_token_text)))
                    g.add((next_uri, EX.iteration, Literal(step + 1)))
                    # Link the current token to the next token
                    g.add((current_uri, EX.next, next_uri))

                    # Save this new token sequence for the next step
                    next_nodes.append(new_input_ids)

        # Update current_nodes for the next iteration to only include exactly top_k branches
        current_nodes = next_nodes

    # Save the entire RDF graph to a single Turtle file
    g.serialize(destination=output_file, format="turtle")
    print(f"Saved branching structure to RDF file: {output_file}")


In [ ]:
# Example usage
prompt = "Once upon a time in a distant land"
output = generate_and_save_rdf(prompt, top_k=3, n_steps=3)

Saved branching structure to RDF file: branching_structure.ttl


# Test 2

In [26]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import nltk

nltk.download('punkt')

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.48s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [ ]:
# Coherence calculation function
def coherence_metrics(text, window_size=4096):
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids

    # Short and long context predictions
    short_context = input_ids[:, :window_size // 2]
    long_context = input_ids

    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
        outputs_long = model(long_context, labels=long_context)

    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    coherence_acc_long = (outputs_long.logits.argmax(-1) == long_context).float().mean().item()

    # Avoid division by zero
    coherence_diff = (coherence_acc_long - coherence_acc_short) / coherence_acc_long if coherence_acc_long != 0 else 0
    return coherence_acc_long, coherence_acc_short, coherence_diff

# Function to generate RDF with different classes
def generate_and_save_rdf_with_classes(prompt, top_k=3, n_steps=5, output_file="branching_structure_with_classes.ttl"):
    # Initialize an RDF graph
    g = Graph()
    EX = Namespace("http://example.org/")
    g.bind("ex", EX)

    # Tokenize the initial prompt
    tokens = tokenizer(prompt, return_tensors='pt')
    input_ids = tokens.input_ids

    # Track each token generated in each step
    current_nodes = [input_ids]

    for step in range(n_steps):
        print(f"\nStep {step + 1}/{n_steps}", end=" ")
        next_nodes = []

        for branch_id, branch_input_ids in enumerate(current_nodes):
            print(f"Branch {branch_id + 1}/{len(current_nodes)}", end=" ")
            with torch.no_grad():
                outputs = model(branch_input_ids)
                logits = outputs.logits[:, -1, :]
                
                # Get top-k tokens
                top_k_probs, top_k_indices = torch.topk(logits, top_k, dim=-1)

                # Create branches for each top-k token
                for k in range(top_k):
                    print(f"Token {k + 1}/{top_k}", end=" ")
                    next_token = top_k_indices[0, k].item()
                    next_token_text = tokenizer.decode([next_token], skip_special_tokens=True)
                    new_input_ids = torch.cat([branch_input_ids, torch.tensor([[next_token]])], dim=-1)

                    # Text and coherence info
                    current_text = tokenizer.decode(branch_input_ids[0], skip_special_tokens=True)
                    next_text = next_token_text.strip()
                    coherence_acc_long, coherence_acc_short, coherence_diff = coherence_metrics(current_text)

                    # URIs for the current and next nodes
                    current_path_uri = EX[f"Path_{step}_{branch_id}"]
                    next_token_uri = EX[f"Token_{step+1}_{branch_id}_{k}"]

                    # Full sequence (Path) representation
                    g.add((current_path_uri, RDF.type, EX.Path))
                    g.add((current_path_uri, EX.text, Literal(current_text)))
                    g.add((current_path_uri, EX.coherenceScore, Literal(coherence_acc_short)))
                    g.add((current_path_uri, EX.iteration, Literal(step)))
                    g.add((current_path_uri, EX.next, next_token_uri))

                    # Individual token representation
                    g.add((next_token_uri, RDF.type, EX.Token))
                    g.add((next_token_uri, EX.text, Literal(next_text)))
                    g.add((next_token_uri, EX.iteration, Literal(step + 1)))

                    # Link between tokens within the token level structure
                    g.add((current_path_uri, EX.next, next_token_uri))

                    # Save new sequence for the next iteration
                    next_nodes.append(new_input_ids)

        # Update nodes for the next step
        current_nodes = next_nodes

    # Save the entire RDF graph to a Turtle file
    g.serialize(destination=output_file, format="turtle")
    print(f"Saved branching structure with classes to RDF file: {output_file}")

In [28]:
# Example usage
prompt = "Once upon a time"
generate_and_save_rdf_with_classes(prompt, top_k=3, n_steps=3, output_file="branching_structure_with_classes.ttl")

Step 1/3
Branch 1/1
Token 1/3
Token 2/3
Token 3/3
Step 2/3
Branch 1/3
Token 1/3
Token 2/3
Token 3/3
Branch 2/3
Token 1/3
Token 2/3
Token 3/3
Branch 3/3
Token 1/3
Token 2/3
Token 3/3
Step 3/3
Branch 1/9
Token 1/3
Token 2/3
Token 3/3
Branch 2/9
Token 1/3
Token 2/3
Token 3/3
Branch 3/9
Token 1/3
Token 2/3
Token 3/3
Branch 4/9
Token 1/3
Token 2/3
Token 3/3
Branch 5/9
Token 1/3
Token 2/3
Token 3/3
Branch 6/9
Token 1/3
Token 2/3
Token 3/3
Branch 7/9
Token 1/3
Token 2/3
Token 3/3
Branch 8/9
Token 1/3
Token 2/3
Token 3/3
Branch 9/9
Token 1/3
Token 2/3
Token 3/3
Saved branching structure with classes to RDF file: branching_structure_with_classes.ttl
